# Automated Training of Custom openWakeWord Models
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Oreozisbored/openWakeWord/blob/modernize-training-notebook/notebooks/automatic_model_training.ipynb)

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated pipeline for synthetic dataset generation, audio augmentation, and model training.


## 1. Environment Setup
Install required system packages, configure an isolated Python 3.11 environment, and install dependencies.


In [ ]:
# @title Install System Packages and Set Up Python Environment { display-mode: "form" }
# @markdown Run this cell to install system dependencies, configure Python 3.11 via uv, and apply compatibility patches.
import sys
import os
import glob
import shutil
import subprocess

print("Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq espeak-ng libespeak-ng-dev ffmpeg git curl

print("Configuring Python 3.11 environment...")
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Detect uv executable path across environments
uv_candidates = [
    shutil.which("uv"),
    os.path.expanduser("~/.local/bin/uv"),
    os.path.expanduser("~/.cargo/bin/uv"),
    "/root/.local/bin/uv",
    "/root/.cargo/bin/uv",
]
UV_BIN = next((c for c in uv_candidates if c and os.path.exists(c)), "uv")

!{UV_BIN} python install 3.11
!{UV_BIN} venv /content/py311 --python 3.11

if not os.path.exists("/content/py311/bin/python"):
    raise RuntimeError("Failed to create Python 3.11 virtual environment at /content/py311. Check that uv installed correctly.")

PY_ENV = "/content/py311/bin/python"
PIP_CMD = f"{UV_BIN} pip install --python /content/py311"

print("Installing PyTorch with CUDA...")
!{PIP_CMD} torch torchaudio --index-url https://download.pytorch.org/whl/cu121

print("Installing dependencies...")
!{PIP_CMD} "scipy==1.13.1" "setuptools<70" "datasets<3" "pyarrow<17" \
    mutagen soundfile audiomentations torch-audiomentations acoustics torchinfo torchmetrics \
    speechbrain deep-phonemizer piper-phonemize espeak-phonemizer webrtcvad pronouncing \
    onnx onnxruntime-gpu onnx2tf tensorflow tensorflow-probability torchcodec onnxscript psutil \
    matplotlib matplotlib-inline

if not os.path.exists("/content/openWakeWord"):
    !git clone https://github.com/dscripka/openWakeWord.git /content/openWakeWord
if not os.path.exists("/content/piper-sample-generator"):
    !git clone https://github.com/dscripka/piper-sample-generator.git /content/piper-sample-generator

!{PIP_CMD} --no-deps -e /content/openWakeWord

# Universal compatibility patch for PyTorch 2.6+, Torchaudio, and headless Matplotlib across all sub-processes
sitecustomize_code = """
import os
os.environ["MPLBACKEND"] = "Agg"

import torch
import torch.serialization
import torchaudio
import soundfile as sf
from collections import namedtuple

_orig_load = torch.load
def _custom_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _custom_load
torch.serialization.load = _custom_load

AudioMetaData = namedtuple("AudioMetaData", ["sample_rate", "num_frames", "num_channels", "bits_per_sample", "encoding"])
def _custom_info(filepath, *args, **kwargs):
    info = sf.info(filepath)
    return AudioMetaData(info.samplerate, info.frames, info.channels, 16, "PCM_S")
torchaudio.info = _custom_info
"""

site_pkgs = glob.glob("/content/py311/lib/python3.11/site-packages")
if site_pkgs:
    with open(os.path.join(site_pkgs[0], "sitecustomize.py"), "w") as f:
        f.write(sitecustomize_code)

with open("/content/openWakeWord/openwakeword/__init__.py", "r") as f:
    init_code = f.read()

if "_custom_load" not in init_code:
    with open("/content/openWakeWord/openwakeword/__init__.py", "w") as f:
        f.write(sitecustomize_code + "\n" + init_code)

# Fix torchaudio reverberation sample rate collision in data.py
data_path = "/content/openWakeWord/openwakeword/data.py"
with open(data_path, "r") as f:
    data_content = f.read()

if "rir_waveform, sr = torchaudio.load" in data_content:
    data_content = data_content.replace(
        "rir_waveform, sr = torchaudio.load",
        "rir_waveform, _ = torchaudio.load"
    )
    with open(data_path, "w") as f:
        f.write(data_content)

# Multi-core augmentation patch in data.py
if "from multiprocessing.pool import ThreadPool" not in data_content:
    with open(data_path, "r") as f:
        dlines = f.readlines()
    for idx, line in enumerate(dlines):
        if "def augment_clips" in line:
            indent = "    "
            inject_patch = [
                f"{indent}from multiprocessing.pool import ThreadPool\n",
                f"{indent}pool_workers = os.cpu_count() or 4\n",
                f"{indent}def _process_single_clip(clip):\n",
                f"{indent}    try:\n",
                f"{indent}        aud, sr = torchaudio.load(clip)\n",
                f"{indent}        if aud.shape[1] > sr:\n",
                f"{indent}            aud = aud[:, :sr]\n",
                f"{indent}        if self.augment_first_pass is not None:\n",
                f"{indent}            aud = self.augment_first_pass(samples=aud.numpy(), sample_rate=sr)\n",
                f"{indent}            aud = torch.tensor(aud)\n",
                f"{indent}        return aud\n",
                f"{indent}    except Exception:\n",
                f"{indent}        return None\n",
                f"{indent}with ThreadPool(processes=pool_workers) as pool:\n",
                f"{indent}    results = pool.map(_process_single_clip, batch)\n",
                f"{indent}clips = [c for c in results if c is not None]\n",
                f"{indent}if not clips: return None\n",
                f"{indent}return torch.vstack(clips)\n"
            ]
            dlines.insert(idx + 1, "".join(inject_patch))
            break
    with open(data_path, "w") as f:
        f.writelines(dlines)

# Fix train.py convert_to_tflite default and CPU threading
train_path = "/content/openWakeWord/openwakeword/train.py"
with open(train_path, "r") as f:
    train_content = f.read()

train_content = train_content.replace('default="False"', 'default=False')
train_content = train_content.replace('ncpu = 1', 'ncpu = n_cpus')

with open(train_path, "w") as f:
    f.write(train_content)

print("Environment setup and patches complete.")


## 2. Hardware Detection
Detect available GPU accelerator and CPU cores to automatically tune batch sizes and worker threads.


In [ ]:
# @title Inspect Hardware Environment and Auto-Tune Performance { display-mode: "form" }
# @markdown Query hardware specifications to auto-scale batch sizes and worker threads.
import os
import psutil
import torch

cpu_count = os.cpu_count() or 1
cuda_available = torch.cuda.is_available()

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if gpu_vram_gb >= 30:
        auto_tts_batch, auto_aug_batch = 150, 64
    elif gpu_vram_gb >= 15:
        auto_tts_batch, auto_aug_batch = 100, 32
    else:
        auto_tts_batch, auto_aug_batch = 50, 16
else:
    gpu_name = "None (CPU)"
    gpu_vram_gb = 0
    auto_tts_batch, auto_aug_batch = 16, 16

print(f"CPU Cores: {cpu_count}")
print(f"System RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"GPU Accelerator: {gpu_name}")
if cuda_available:
    print(f"VRAM: {gpu_vram_gb:.1f} GB")
print(f"Auto-Tuned TTS Batch Size: {auto_tts_batch}")
print(f"Auto-Tuned Augmentation Batch Size: {auto_aug_batch}")


## 3. Download Models and Checkpoints
Download base feature extraction models, DeepPhonemizer, and the Piper TTS voice model.


In [ ]:
# @title Download Models and Checkpoints { display-mode: "form" }
# @markdown Fetch openWakeWord base feature extractors, DeepPhonemizer, and high-fidelity Piper TTS model.
import os
import urllib.request
import shutil

# Verify Python 3.11 environment exists
if not os.path.exists("/content/py311/bin/python"):
    raise RuntimeError("Python 3.11 environment not found at /content/py311. Please run Step 1 (Environment Setup) first!")

# Setup opener with standard User-Agent for downloads
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "Mozilla/5.0")]
urllib.request.install_opener(opener)

py311_code = """
import openwakeword
from openwakeword.utils import download_models
download_models()
"""
with open("/content/dl_models.py", "w") as f:
    f.write(py311_code)
!/content/py311/bin/python /content/dl_models.py

# Download DeepPhonemizer checkpoint from verified Hugging Face mirror
os.makedirs("/content/openWakeWord/openwakeword/resources", exist_ok=True)
dp_dest = "/content/openWakeWord/openwakeword/resources/en_us_cmudict_forward.pt"
if not os.path.exists(dp_dest) or os.path.getsize(dp_dest) < 1000:
    print("Downloading DeepPhonemizer checkpoint...")
    dp_url = "https://huggingface.co/TaterTotterson/DeepPhonemizer/resolve/main/en_us_cmudict_forward.pt"
    urllib.request.urlretrieve(dp_url, dp_dest)

# Download Piper high-fidelity LibriTTS voice checkpoint
os.makedirs("/content/piper-sample-generator/models", exist_ok=True)
piper_model = "/content/piper-sample-generator/models/en-us-libritts-high.pt"
if not os.path.exists(piper_model) or os.path.getsize(piper_model) < 1000000:
    print("Downloading Piper high-fidelity voice model...")
    piper_url = "https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt"
    urllib.request.urlretrieve(piper_url, piper_model)

piper_json = "/content/piper-sample-generator/models/en-us-libritts-high.pt.json"
if not os.path.exists(piper_json):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/dscripka/piper-sample-generator/master/models/en-us-libritts-high.pt.json",
        piper_json
    )

print("Pretrained models and checkpoints ready.")


## 4. Download Audio and Features
Download pre-computed ACAV100M features, validation datasets, room impulse responses, and environmental background noise.


In [ ]:
# @title Download Validation and Background Audio Datasets { display-mode: "form" }
# @markdown Download pre-computed acoustic features (~16 GB) and stream MIT impulse responses and ESC-50 noise.
import os

# 1. Validation features (~176 MB)
if not os.path.exists("/content/validation_set_features.npy"):
    print("Downloading validation set features...")
    !curl -L -s -o /content/validation_set_features.npy "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy"

# 2. ACAV100M pre-computed 2,000 hour features (~16 GB)
if not os.path.exists("/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    print("Downloading ACAV100M training features (~16 GB)...")
    !curl -L -s -o /content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"

# 3. Stream MIT Room Impulse Responses and ESC-50 noise clips
dl_audio_script = """
import os, soundfile as sf
from datasets import load_dataset
from tqdm import tqdm

os.makedirs("/content/mit_rirs", exist_ok=True)
if len(os.listdir("/content/mit_rirs")) < 50:
    print("Fetching MIT Room Impulse Responses...")
    rir_ds = load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for i, row in enumerate(tqdm(rir_ds, total=271)):
        sf.write(f"/content/mit_rirs/{i}.wav", row["audio"]["array"], row["audio"]["sampling_rate"])

os.makedirs("/content/background_clips", exist_ok=True)
if len(os.listdir("/content/background_clips")) < 100:
    print("Fetching ESC-50 background noise clips...")
    esc_ds = load_dataset("ashraq/esc50", split="train", streaming=True)
    for i, row in enumerate(tqdm(esc_ds, total=2000)):
        sf.write(f"/content/background_clips/{i}.wav", row["audio"]["array"], row["audio"]["sampling_rate"])
"""

with open("/content/dl_audio.py", "w") as f:
    f.write(dl_audio_script)
!/content/py311/bin/python /content/dl_audio.py
print("Audio datasets ready.")


## 5. Training Configuration
Define the target wake word and training parameters. Phonetic spellings (such as `kuhm puter`) can be used to improve pronunciation accuracy.


In [ ]:
# @title Wake Word Training Configuration { display-mode: "form" }
# @markdown Configure wake word phrase and training hyperparameters.
import os
import yaml

target_wake_word = "kuhm puter" # @param {type:"string"}
number_of_examples = 10000 # @param {type:"slider", min:1000, max:50000, step:1000}
number_of_training_steps = 10000 # @param {type:"slider", min:1000, max:50000, step:1000}
false_activation_penalty = 1000 # @param {type:"slider", min:100, max:5000, step:100}

# Auto-tuned parameters from hardware detection
tts_batch_size = auto_tts_batch if 'auto_tts_batch' in locals() else 100
augmentation_batch_size = auto_aug_batch if 'auto_aug_batch' in locals() else 32

model_clean_name = target_wake_word.strip().replace(" ", "_")
output_dir = f"/content/model_{model_clean_name}"
os.makedirs(output_dir, exist_ok=True)

config = {
    "model_name": model_clean_name,
    "target_phrase": [target_wake_word],
    "custom_negative_phrases": [],
    "n_samples": number_of_examples,
    "n_samples_val": max(500, number_of_examples // 10),
    "tts_batch_size": tts_batch_size,
    "augmentation_batch_size": augmentation_batch_size,
    "piper_sample_generator_path": "/content/piper-sample-generator",
    "output_dir": output_dir,
    "rir_paths": ["/content/mit_rirs"],
    "background_paths": ["/content/background_clips"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "/content/validation_set_features.npy",
    "augmentation_rounds": 1,
    "feature_data_files": {
        "ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    },
    "batch_n_per_class": {
        "ACAV100M_sample": 1024,
        "adversarial_negative": 50,
        "positive": 50
    },
    "model_type": "dnn",
    "layer_size": 32,
    "steps": number_of_training_steps,
    "max_negative_weight": false_activation_penalty,
    "target_false_positives_per_hour": 0.2,
    "target_accuracy": 0.5,
    "target_recall": 0.25
}

yaml_file = f"/content/{model_clean_name}.yaml"
with open(yaml_file, "w") as f:
    yaml.dump(config, f)

print(f"Configuration saved to {yaml_file}")
print(f"Target Phrase: {target_wake_word}")
print(f"Positive Samples: {number_of_examples}")
print(f"Training Steps: {number_of_training_steps}")
print(f"False Activation Penalty: {false_activation_penalty}")
print(f"TTS Batch Size: {tts_batch_size} | Augmentation Batch Size: {augmentation_batch_size}")


## 6. Test Pronunciation
Synthesize sample audio clips with Piper TTS and listen to them to verify pronunciation before running full training.


In [ ]:
# @title Test Pronunciation { display-mode: "form" }
# @markdown Synthesize sample audio clips with Piper TTS and listen to verify pronunciation before training.
import os
import shutil
import glob
from IPython.display import Audio, display

test_phrase = target_wake_word if 'target_wake_word' in locals() else "kuhm puter" # @param {type:"string"}
test_samples = 3 # @param {type:"slider", min:1, max:5, step:1}

test_dir = "/content/test_pronunciation"
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)
os.makedirs(test_dir, exist_ok=True)

print(f"Synthesizing {test_samples} test clip(s) for '{test_phrase}'...")
!/content/py311/bin/python /content/piper-sample-generator/generate_samples.py \
    "{test_phrase}" \
    --max-samples {test_samples} \
    --model /content/piper-sample-generator/models/en-us-libritts-high.pt \
    --output-dir {test_dir}

wav_files = sorted(glob.glob(f"{test_dir}/*.wav"))
if wav_files:
    print(f"Generated {len(wav_files)} clip(s):")
    for i, wav_path in enumerate(wav_files):
        print(f"Sample {i+1}:")
        display(Audio(wav_path, autoplay=False))
else:
    print("Could not generate test clips. Ensure Pretrained Models cell has been executed.")


## 7. Run Training Pipeline
Execute data generation, clip augmentation, and model training.


In [ ]:
# @title Train the Model { display-mode: "form" }
# @markdown Run clip generation, feature augmentation, and DNN model training.
import os
import subprocess
import sys

# Verify Python 3.11 environment exists
if not os.path.exists("/content/py311/bin/python"):
    raise RuntimeError("Python 3.11 environment not found at /content/py311. Please run Step 1 (Environment Setup) first!")

yaml_file = f"/content/{model_clean_name}.yaml"
PY_BIN = "/content/py311/bin/python"

sub_env = os.environ.copy()
sub_env["MPLBACKEND"] = "Agg"

def run_phase(cmd, phase_name):
    print(f"\n=== {phase_name} ===")
    res = subprocess.run(cmd, shell=True, env=sub_env)
    if res.returncode != 0:
        raise RuntimeError(f"{phase_name} failed with exit code {res.returncode}. See output above.")

run_phase(f"{PY_BIN} /content/openWakeWord/openwakeword/train.py --training_config {yaml_file} --generate_clips", "Phase 1: Generating synthetic audio clips")
run_phase(f"{PY_BIN} /content/openWakeWord/openwakeword/train.py --training_config {yaml_file} --augment_clips", "Phase 2: Augmenting clips and computing features")
run_phase(f"{PY_BIN} /content/openWakeWord/openwakeword/train.py --training_config {yaml_file} --train_model", "Phase 3: Training model")

onnx_expected = os.path.join(output_dir, f"{model_clean_name}.onnx")
if not os.path.exists(onnx_expected):
    raise FileNotFoundError(f"Training completed but expected ONNX model was not found at {onnx_expected}.")

print(f"\nModel training successfully finished! ONNX Model: {onnx_expected}")


## 8. Convert to TFLite
Convert the trained ONNX model to TFLite format for deployment on Home Assistant or embedded hardware.


In [ ]:
# @title Convert Model to TFLite { display-mode: "form" }
# @markdown Convert the trained ONNX model to TFLite using onnx2tf with input flattening.
import os
import shutil

onnx_path = os.path.join(output_dir, f"{model_clean_name}.onnx")
dest_tflite = os.path.join(output_dir, f"{model_clean_name}.tflite")
float32_tflite = os.path.join(output_dir, f"{model_clean_name}_float32.tflite")

if not os.path.exists(onnx_path):
    raise FileNotFoundError(f"ONNX model not found at {onnx_path}. Ensure Step 7 (Train the Model) completed successfully without errors!")

print(f"Converting {onnx_path} to TFLite...")
!/content/py311/bin/onnx2tf -i {onnx_path} -o {output_dir}/ -kat onnx____Flatten_0

if os.path.exists(float32_tflite):
    shutil.move(float32_tflite, dest_tflite)

if not os.path.exists(dest_tflite):
    raise RuntimeError(f"TFLite conversion failed to produce {dest_tflite}.")

print(f"TFLite Model: {dest_tflite} ({os.path.getsize(dest_tflite)} bytes)")
print(f"ONNX Model: {onnx_path} ({os.path.getsize(onnx_path)} bytes)")


## 9. Test Inference and Download
Verify that the trained models load successfully and download the ONNX and TFLite files.


In [ ]:
# @title Test Inference and Download Models { display-mode: "form" }
# @markdown Verify inference on a silent test frame and download the output models.
import os

onnx_path = os.path.join(output_dir, f"{model_clean_name}.onnx")
tflite_path = os.path.join(output_dir, f"{model_clean_name}.tflite")

verify_code = f"""
import numpy as np
import openwakeword
from openwakeword.model import Model

onnx_path = "{onnx_path}"
tflite_path = "{tflite_path}"

if __import__("os").path.exists(onnx_path):
    print("Verifying ONNX inference...")
    oww_onnx = Model(wakeword_models=[onnx_path], inference_framework="onnx")
    pred = oww_onnx.predict(np.zeros(1280, dtype=np.int16))
    print("ONNX prediction on silent frame:", pred)

if __import__("os").path.exists(tflite_path):
    print("Verifying TFLite inference...")
    oww_tflite = Model(wakeword_models=[tflite_path], inference_framework="tflite")
    pred_tf = oww_tflite.predict(np.zeros(1280, dtype=np.int16))
    print("TFLite prediction on silent frame:", pred_tf)
"""

with open("/content/test_inference.py", "w") as f:
    f.write(verify_code)

!/content/py311/bin/python /content/test_inference.py

print("\nDownloading models...")
try:
    from google.colab import files
    if os.path.exists(onnx_path):
        files.download(onnx_path)
    if os.path.exists(tflite_path):
        files.download(tflite_path)
except Exception:
    print(f"Models are located in: {output_dir}")
